# **Data Description**

The dataset contains concrete images having cracks. The data is collected from various METU Campus Buildings.
The dataset is divided into two as negative and positive crack images for image classification.
Each class has 20000images with a total of 40000 images with 227 x 227 pixels with RGB channels.
The dataset is generated from 458 high-resolution images (4032x3024 pixel) with the method proposed by Zhang et al (2016).
High-resolution images have variance in terms of surface finish and illumination conditions.
No data augmentation in terms of random rotation or flipping is applied.

# **Data Gathering**

In [1]:
# Import libraries

import os
import warnings
warnings.filterwarnings('ignore')
import cv2
import random
from joblib import Parallel, delayed
from PIL import Image 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.models import Model
from keras.preprocessing.image import img_to_array, load_img
from tensorflow.keras.layers import LSTM, Dense, Input, GlobalAveragePooling2D, Lambda

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

ModuleNotFoundError: No module named 'cv2'

In [ ]:
# Collect data
negative_dir = '/kaggle/input/concrete-crack-images-for-classification/Negative'
positive_dir = '/kaggle/input/concrete-crack-images-for-classification/Positive'

In [ ]:
def load_images(folder):
    images = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        if img_path.endswith('.jpg') or img_path.endswith('.png'):
            img = Image.open(img_path)
            images.append(img)
    return images

In [ ]:
negative_images = load_images(negative_dir)
positive_images = load_images(positive_dir)

In [ ]:
# Display samples of Negative and Positive Concrete Cracks
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.subplot(2, 4, i + 1)
    plt.imshow(negative_images[i])
    plt.axis('off')
    plt.title('Negative')
    
    plt.subplot(2, 4, i + 5)
    plt.imshow(positive_images[i])
    plt.axis('off')
    plt.title('Positive')

plt.tight_layout()
plt.show()

# **Preprocessing**

In [ ]:
img_size = (224, 224)

# Create data generators
data_gen = ImageDataGenerator(rescale=1.0/255.0, validation_split=0.2)

train_gen = data_gen.flow_from_directory(
    '/kaggle/input/concrete-crack-images-for-classification',
    target_size=img_size,
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_gen = data_gen.flow_from_directory(
    '/kaggle/input/concrete-crack-images-for-classification',
    target_size=img_size,
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

In [ ]:
def load_images_with_edges(folder, size=(112, 112)):
    images = []
    edges = []
    for filename in os.listdir(folder):
        img_path = os.path.join(folder, filename)
        if img_path.endswith('.jpg') or img_path.endswith('.png'):
            img = Image.open(img_path)
            img = img.resize(size)  
            img_array = np.array(img)
            gray_img = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY) 
            
            # Perform Canny edge detection
            edge_img = cv2.Canny(gray_img, threshold1=100, threshold2=200)
            
            # Convert edge image back to PIL for visualization
            edges.append(Image.fromarray(edge_img))
            images.append(img)
    return images, edges

In [ ]:
negative_images, negative_edges = load_images_with_edges(negative_dir)
positive_images, positive_edges = load_images_with_edges(positive_dir)

In [ ]:
# Visualize edges for Concrete
plt.figure(figsize=(10, 10))
for i in range(4):
    # Original negative images
    plt.subplot(4, 4, i + 1)
    plt.imshow(negative_images[i])
    plt.axis('off')
    plt.title('Negative Image')
    
    # Canny edges for negative images
    plt.subplot(4, 4, i + 5)
    plt.imshow(negative_edges[i], cmap='gray')
    plt.axis('off')
    plt.title('Negative Edges')
    
    # Original positive images
    plt.subplot(4, 4, i + 9)
    plt.imshow(positive_images[i])
    plt.axis('off')
    plt.title('Positive Image')
    
    # Canny edges for positive images
    plt.subplot(4, 4, i + 13)
    plt.imshow(positive_edges[i], cmap='gray')
    plt.axis('off')
    plt.title('Positive Edges')

plt.tight_layout()
plt.show()

# **CNN Model**

In [ ]:
from tensorflow.keras import layers, models

# Define the CNN model
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_gen, epochs=10, validation_data=val_gen)

In [ ]:
# Visualize accuracy and model loss 
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

# **Pretrained Model**

In [ ]:
# Extract deep features using the VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(train_gen, epochs=10, validation_data=val_gen)

In [ ]:
val_loss, val_accuracy = model.evaluate(val_gen)
print(f'Validation accuracy: {val_accuracy:.2f}')

# **Prediction**

In [ ]:
def predict_image(img_path):
    img = Image.open(img_path)
    img = img.resize(img_size)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    prediction = model.predict(img_array)
    return 'Positive' if prediction[0][0] > 0.5 else 'Negative'

# Example prediction (by positive crack)
print(predict_image('/kaggle/input/concrete-crack-images-for-classification/Positive/00004.jpg'))

In [ ]:
# Example prediction (by negative crack)
print(predict_image('/kaggle/input/concrete-crack-images-for-classification/Negative/00002.jpg'))